# 1.11 · dbt 入门 / dbt Intro —— Part 1 收官 🏁

> **课程定位 / Where this fits**
> **Part 1 最后一课**。1.10 说现代栈是 ELT，"T" 用 SQL 在仓内完成——**dbt（data build tool）就是管理这些 SQL 的工具**。它把 1.12 节工程化的那套（git / 测试 / 模块化）带进了 SQL 世界。
> **Part 1 finale.** dbt manages the "T" in ELT — bringing the Part 0.12 engineering discipline (git / tests / modularity) into SQL.

> ⚠ **本节不安装 dbt**——而是**用 ~50 行 Python 实现一个 mini-dbt**，跑通 model / ref / DAG / test 的完整闭环。理解了引擎，真 dbt 的 CLI 半天就能上手。
> We don't install dbt — we **build a mini-dbt in ~50 lines of Python** and run the full model/ref/DAG/test loop. Understand the engine and the real CLI takes an afternoon.

> 💡 **面试相关 / Interview-relevant**
> - "dbt 解决什么问题" ★★★★（分析工程岗必考）
> - "`ref()` 为什么重要" ★★★★
> - "staging → intermediate → marts 分层" ★★★★
> - "dbt test 有哪几种" ★★★
> - "incremental model 什么时候用" ★★★

---

## 学习目标 / Learning Objectives

1. 说出 dbt 之前 SQL 转换管理的**四大痛点**，以及 dbt 各用什么解决。
   Name the four pre-dbt pains and dbt's answer to each.
2. 理解 **model = 一个 SELECT 文件**，物化策略（view / table / incremental）怎么选。
   Model = one SELECT file; pick view / table / incremental materializations.
3. 解释 **`ref()`** 如何同时解决依赖排序和环境切换。
   Explain how `ref()` solves both dependency ordering and environment switching.
4. 写出 **staging → intermediate → marts** 三层架构并说明各层职责。
   Lay out the three-layer architecture and each layer's job.
5. 用 mini-dbt **跑通一个真实 DAG**：解析 ref → 拓扑排序 → 物化 → 测试。
   Run a real DAG end-to-end in our mini-dbt.
6. 知道 4 种内置测试和写法。
   Know the four built-in tests.

---

## 目录 / Table of Contents

1. [dbt 之前的世界 / The World Before dbt](#1)
2. [dbt 是什么 / What dbt Is (and Isn't)](#2)
3. [核心概念：model 与 ref() ⭐](#3)
4. [动手：50 行实现 mini-dbt ⭐ / Build a Mini-dbt](#4)
5. [三层架构：staging → intermediate → marts ⭐](#5)
6. [用 mini-dbt 跑通完整项目 / Run the Full Project](#6)
7. [测试 / Tests](#7)
8. [物化策略 / Materializations](#8)
9. [真实 dbt 项目长什么样 / A Real dbt Project](#9)
10. [小结 + Part 1 总结 🏁 / Summary + Part 1 Wrap-up](#10)


<a id="1"></a>
## 1. dbt 之前的世界 / The World Before dbt

2016 年前，数仓转换层的真实样子：
What warehouse transformation looked like before 2016:

```
/shared_drive/sql_scripts/
├── daily_revenue_v2_FINAL.sql
├── daily_revenue_v2_FINAL_fixed.sql        ← 哪个是对的？
├── customer_dim_DO_NOT_DELETE.sql
└── run_all.sh                              ← 顺序靠手工维护
```

| 痛点 / Pain | 后果 / Consequence | dbt 的解法 / dbt's answer |
|---|---|---|
| **依赖靠人记** | 改了上游忘了跑下游 → 报表错数 | `ref()` 自动构建 DAG，按序执行 |
| **没有测试** | NULL / 重复键悄悄混进报表 | `dbt test` 声明式数据测试 |
| **没有版本控制** | "v2_FINAL_fixed" 地狱 | model = 文件 → git / PR / code review |
| **环境混乱** | 开发直接改生产表 | 同一份代码编译到 dev / prod schema |

**一句话**：dbt 把软件工程纪律（Part 0.12 那一套）搬进了 SQL 转换层。
One line: dbt brings software-engineering discipline into the SQL transformation layer.


<a id="2"></a>
## 2. dbt 是什么 / What dbt Is (and Isn't)

### 是 / It IS

- 一个**编译器 + 编排器**：把带 `ref()` / Jinja 模板的 SQL 编译成纯 SQL，按依赖顺序发给数仓执行
- **数仓内**工具：计算全部发生在 Snowflake / BigQuery / DuckDB 里，dbt 自己不算数据

### 不是 / It is NOT

- ❌ 不是数据库（不存数据）
- ❌ 不是 EL 工具（不搬数据进仓——那是 Fivetran / Airbyte 的事）
- ❌ 不是调度器（定时跑要靠 Airflow / dbt Cloud / cron）

### 工作流 / The workflow

```
你写:    models/marts/fct_sales.sql        ← 一个 SELECT（带 ref()）
dbt 做:  1. 解析所有 model 的 ref() → 构建 DAG
         2. Jinja 编译 → 纯 SQL
         3. 包成 CREATE TABLE/VIEW AS ... 按拓扑序发给数仓
         4. 跑 schema 测试
你得到:  数仓里一套有序、可测、有血缘的表
```

下面我们**自己把这四步写出来**——这是理解 dbt 最快的方式。
We'll now write those four steps ourselves — the fastest way to truly understand dbt.


<a id="3"></a>
## 3. 核心概念：model 与 ref() ⭐

### Model = 一个 SELECT 文件

```sql
-- models/staging/stg_invoices.sql
SELECT
    invoice_id,
    customer_id,
    track_id,
    invoice_date,
    quantity
FROM raw.invoice
WHERE quantity > 0          -- 清洗规则住在这里 / cleaning rules live here
```

**没有 CREATE TABLE**——dbt 自动包上。你只声明"这个 model 的内容是什么"。
No CREATE TABLE — dbt wraps it. You declare *what* the model contains.

### `ref()` = dbt 的灵魂 ⭐

```sql
-- models/marts/fct_sales.sql
SELECT ...
FROM {{ ref('stg_invoices') }} i        -- ← 不写死表名！/ never hard-code table names
JOIN {{ ref('stg_tracks') }} t USING (track_id)
```

`ref()` 一石二鸟 / two birds, one stone:

1. **依赖图**：dbt 扫出 `fct_sales` 依赖 `stg_invoices` 和 `stg_tracks` → 自动先跑上游（拓扑排序）
2. **环境切换**：同一份代码，dev 编译成 `dev_alice.stg_invoices`，prod 编译成 `analytics.stg_invoices`——**开发永远不会误伤生产**

> 💡 **面试一句话答 "`ref()` 为什么重要"**：
> "It turns table references into graph edges — giving you automatic dependency ordering, environment isolation, and data lineage from the same one mechanism."


<a id="4"></a>
## 4. 动手：50 行实现 mini-dbt ⭐ / Build a Mini-dbt

实现 dbt 的三个核心机制：**ref 解析 → 拓扑排序 → 物化执行**。
The three core mechanisms: ref resolution → topological sort → materialization.


In [ ]:
import re
import duckdb
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

class MiniDbt:
    """一个教学版 dbt：models 注册为 (名字, SQL, 物化方式)，
    ref() 解析依赖，拓扑排序后按序物化到 DuckDB。"""

    REF = re.compile(r"\{\{\s*ref\('(\w+)'\)\s*\}\}")

    def __init__(self, conn, schema="analytics"):
        self.conn = conn
        self.schema = schema
        self.models = {}                      # name -> (sql, materialization)
        conn.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")

    def model(self, name, sql, materialized="view"):
        self.models[name] = (sql, materialized)

    def _deps(self, name):
        # 步骤 1: 扫描 SQL 里的 ref() → 依赖边 / scan refs -> graph edges
        return self.REF.findall(self.models[name][0])

    def _topo_order(self):
        # 步骤 2: 拓扑排序（DFS 后序）/ topological sort via DFS
        order, seen = [], set()
        def visit(n, path):
            if n in path:
                raise ValueError(f"循环依赖 / cycle: {' -> '.join(path)} -> {n}")
            if n in seen:
                return
            for dep in self._deps(n):
                visit(dep, path + [n])
            seen.add(n); order.append(n)
        for n in self.models:
            visit(n, [])
        return order

    def run(self):
        # 步骤 3: 编译（ref → 实际表名）并物化 / compile & materialize
        for name in self._topo_order():
            sql, mat = self.models[name]
            compiled = self.REF.sub(rf"{self.schema}.\1", sql)
            kind = "TABLE" if mat == "table" else "VIEW"
            self.conn.sql(f"CREATE OR REPLACE {kind} {self.schema}.{name} AS {compiled}")
            print(f"  ✓ {mat:<5} {self.schema}.{name}   (deps: {self._deps(name) or '—'})")

print("MiniDbt ready —", len([m for m in dir(MiniDbt) if not m.startswith('_')]), "public methods")


**就这么多**。真 dbt 当然还有 Jinja 模板、增量策略、快照、文档站……但**核心引擎就是这三步**：解析 ref → 排序 → 物化。
That's it. Real dbt adds Jinja, incremental strategies, snapshots, docs — but the core engine is these three steps.


<a id="5"></a>
## 5. 三层架构：staging → intermediate → marts ⭐

dbt 社区的标准项目分层（也是面试必考）：
The community-standard layering (and an interview staple):

```
raw (源数据, EL 工具落进来的, 不动它)
 │
 ├── staging/       stg_*     1:1 对应源表：改名、转类型、轻清洗。无 JOIN！
 │
 ├── intermediate/  int_*     可复用的中间逻辑：JOIN、业务规则
 │
 └── marts/         fct_* / dim_*    星型模型成品，给 BI / DS 用
```

| 层 / Layer | 职责 / Job | 规则 / Rules |
|---|---|---|
| **staging** | 标准化源数据 | 一个源表一个 stg；只 SELECT 单表；统一命名/类型 |
| **intermediate** | 复用的业务逻辑块 | 不直接暴露给用户；名字描述动作（`int_orders_joined`）|
| **marts** | 最终事实/维度表 | 1.10 节的星型模型住这里；按业务域分文件夹 |

**为什么分层**：上游源表改了字段名 → 只改对应的一个 stg 文件，下游全部无感。**变更隔离**是分层的全部意义。
Why layers: a source rename touches exactly one stg file; everything downstream is insulated. Change isolation is the whole point.


<a id="6"></a>
## 6. 用 mini-dbt 跑通完整项目 / Run the Full Project

用 Music Store 原始数据建一个**三层 dbt 项目**：raw → staging → marts。
A real three-layer project on the Music Store raw data.


In [ ]:
# ---- 准备 raw 层（模拟 EL 工具落仓的原始数据）----
# The raw layer — what an EL tool would land
conn = duckdb.connect()
conn.sql("""
CREATE SCHEMA raw;

CREATE TABLE raw.invoice AS SELECT * FROM (VALUES
    (1,1,1,DATE '2026-01-05',1),(2,1,9,DATE '2026-01-05',2),
    (3,2,4,DATE '2026-01-10',1),(4,2,18,DATE '2026-01-10',1),
    (5,3,5,DATE '2026-02-12',1),(6,3,6,DATE '2026-02-12',1),
    (7,3,7,DATE '2026-02-12',3),(8,4,15,DATE '2026-02-20',1),
    (9,4,12,DATE '2026-02-20',1),(10,4,13,DATE '2026-02-20',1),
    (11,5,9,DATE '2026-03-01',1),(12,5,10,DATE '2026-03-01',1),
    (13,5,11,DATE '2026-03-01',1),(14,5,4,DATE '2026-03-05',2),
    (15,1,18,DATE '2026-03-15',1),(16,1,19,DATE '2026-03-15',1),
    (17,2,15,DATE '2026-04-01',1),(18,3,14,DATE '2026-04-10',2),
    (19,4,8,DATE '2026-05-02',1),(20,5,1,DATE '2026-05-20',1),
    (21,5,1,DATE '2026-05-21',0)            -- ⚠ 脏数据: qty=0，staging 层要洗掉
) AS t(invoice_id, customer_id, track_id, invoice_date, quantity);

CREATE TABLE raw.track AS SELECT * FROM (VALUES
    (1,'Come Together','Rock',0.99),(2,'Something','Rock',0.99),
    (3,'Here Comes the Sun','Rock',0.99),(4,'Time','Rock',1.29),
    (5,'Money','Rock',1.29),(6,'Us and Them','Rock',1.29),
    (7,'Another Brick in the Wall','Rock',1.29),(8,'Comfortably Numb','Rock',1.29),
    (9,'So What','Jazz',1.49),(10,'Freddie Freeloader','Jazz',1.49),
    (11,'Blue in Green','Jazz',1.49),(12,'One More Time','Electronic',1.29),
    (13,'Aerodynamic','Electronic',1.29),(14,'Digital Love','Electronic',1.29),
    (15,'Get Lucky','Electronic',1.29),(16,'Instant Crush','Electronic',1.29),
    (17,'Lose Yourself to Dance','Electronic',1.29),(18,'Paranoid Android','Rock',1.29),
    (19,'Karma Police','Rock',1.29),(20,'No Surprises','Rock',1.29),
    (21,'Untitled Demo 1','Rock',0.50),(22,'Untitled Demo 2','Rock',0.50)
) AS t(track_id, title, genre, price);

CREATE TABLE raw.customer AS SELECT * FROM (VALUES
    (1,'Alice Chen','US'),(2,'Bob Smith','UK'),(3,'Charlie Davis','US'),
    (4,'Diana Park','DE'),(5,'Ethan Miller','US'),(6,'Fiona Wong','JP')
) AS t(customer_id, name, country);
""")
print("raw layer:", conn.sql("SELECT table_name FROM information_schema.tables WHERE table_schema='raw'").df()["table_name"].tolist())


In [ ]:
# ---- 定义三层 models（注意每层只 ref 上一层）----
dbt = MiniDbt(conn, schema="analytics")

# === staging: 1:1 清洗，无 JOIN / 1:1 cleaning, no JOINs ===
dbt.model("stg_invoices", """
    SELECT invoice_id, customer_id, track_id, invoice_date, quantity
    FROM raw.invoice
    WHERE quantity > 0                      -- 洗掉脏行 / drop dirty rows
""")

dbt.model("stg_tracks", """
    SELECT track_id, title, genre, price
    FROM raw.track
""")

dbt.model("stg_customers", """
    SELECT customer_id, name AS customer_name, country
    FROM raw.customer
""")

# === intermediate: 可复用 JOIN 逻辑 ===
dbt.model("int_sales_enriched", """
    SELECT
        i.invoice_id, i.invoice_date, i.quantity,
        t.title, t.genre, t.price,
        i.quantity * t.price AS revenue,
        c.customer_name, c.country
    FROM {{ ref('stg_invoices') }} i
    JOIN {{ ref('stg_tracks') }} t USING (track_id)
    JOIN {{ ref('stg_customers') }} c USING (customer_id)
""")

# === marts: 业务成品表（物化为 table）===
dbt.model("fct_monthly_revenue", """
    SELECT
        DATE_TRUNC('month', invoice_date) AS month,
        genre,
        SUM(revenue)  AS revenue,
        SUM(quantity) AS units
    FROM {{ ref('int_sales_enriched') }}
    GROUP BY ALL
""", materialized="table")

dbt.model("dim_customer_summary", """
    SELECT
        customer_name,
        country,
        COUNT(DISTINCT invoice_id) AS n_orders,
        ROUND(SUM(revenue), 2)     AS lifetime_value
    FROM {{ ref('int_sales_enriched') }}
    GROUP BY ALL
""", materialized="table")

# ---- dbt run！注意输出顺序：自动按依赖排好 ----
print("=== mini-dbt run ===")
dbt.run()


**看输出顺序**：staging 三兄弟先跑（无依赖）→ `int_sales_enriched`（依赖三个 stg）→ 两个 mart（依赖 int）。**我们没有手写任何顺序**——`ref()` + 拓扑排序自动搞定。这就是 dbt 的核心价值。
Look at the order: the three staging models first, then the intermediate, then the marts — **we never specified any order**. `ref()` + topo-sort did it. That's dbt's core value.


In [ ]:
# 查询成品 mart / Query the finished marts
print("--- fct_monthly_revenue ---")
print(conn.sql("SELECT * FROM analytics.fct_monthly_revenue ORDER BY month, revenue DESC").df())

print("\n--- dim_customer_summary ---")
print(conn.sql("SELECT * FROM analytics.dim_customer_summary ORDER BY lifetime_value DESC").df())


In [ ]:
# 血缘 / Lineage: ref() 的副产品——免费拿到依赖图
print("=== Data lineage (谁依赖谁) ===\n")
for name in dbt._topo_order():
    deps = dbt._deps(name)
    arrow = " ← " + ", ".join(deps) if deps else "   (source: raw.*)"
    print(f"  {name:<24}{arrow}")


真 dbt 把这个图渲染成**交互网页**（`dbt docs serve`）——新人入职看血缘图理解整个数仓，比读 wiki 快得多。
Real dbt renders this as an interactive site (`dbt docs serve`) — new hires learn the warehouse from the lineage graph.


<a id="7"></a>
## 7. 测试 / Tests

dbt 的测试 = **"对这张表跑一条 SQL，返回 0 行 = 通过"**。四个内置测试覆盖 90% 需求：
A dbt test = "run a query against the table; zero rows = pass". Four built-ins cover 90%:

```yaml
# models/marts/schema.yml （真 dbt 的写法 / real dbt syntax）
models:
  - name: dim_customer_summary
    columns:
      - name: customer_name
        tests: [unique, not_null]
      - name: country
        tests:
          - accepted_values: {values: ['US', 'UK', 'DE', 'JP', 'FR', 'CA']}
      - name: customer_id
        tests:
          - relationships: {to: ref('stg_customers'), field: customer_id}
```

| 测试 / Test | 检查 / Checks | 等价 SQL 思路 |
|---|---|---|
| `unique` | 列无重复 | `GROUP BY col HAVING COUNT(*) > 1` |
| `not_null` | 列无 NULL | `WHERE col IS NULL` |
| `accepted_values` | 值在白名单内 | `WHERE col NOT IN (...)` |
| `relationships` | 外键完整性（孤儿检测）| 1.2 节的 anti-join！|

给 mini-dbt 加上测试支持：
Let's add test support to mini-dbt:


In [ ]:
def run_tests(conn, schema, tests):
    # tests: list of (model, test_name, sql_returning_bad_rows)
    print("=== mini-dbt test ===")
    n_fail = 0
    for model, test_name, sql in tests:
        bad = conn.sql(sql.format(t=f"{schema}.{model}")).df()
        status = "PASS ✓" if len(bad) == 0 else f"FAIL ✗ ({len(bad)} bad rows)"
        if len(bad): n_fail += 1
        print(f"  {status:<22} {model} :: {test_name}")
    print(f"\n{'全部通过 🎉' if n_fail == 0 else f'{n_fail} 个测试失败'}")

tests = [
    # unique: 客户名不重复
    ("dim_customer_summary", "unique(customer_name)",
     "SELECT customer_name FROM {t} GROUP BY customer_name HAVING COUNT(*) > 1"),
    # not_null: LTV 无 NULL
    ("dim_customer_summary", "not_null(lifetime_value)",
     "SELECT * FROM {t} WHERE lifetime_value IS NULL"),
    # accepted_values: 国家白名单
    ("dim_customer_summary", "accepted_values(country)",
     "SELECT * FROM {t} WHERE country NOT IN ('US','UK','DE','JP','FR','CA')"),
    # relationships: fct 月份都来自合法日期（示意：无 NULL 月份）
    ("fct_monthly_revenue", "not_null(month)",
     "SELECT * FROM {t} WHERE month IS NULL"),
    # 业务测试: 收入不为负
    ("fct_monthly_revenue", "revenue >= 0",
     "SELECT * FROM {t} WHERE revenue < 0"),
]
run_tests(conn, "analytics", tests)


**真实价值**：这些测试在**每次 `dbt run` 后自动跑**，CI 里 PR 合并前也跑——脏数据在**进报表之前**被拦住，而不是被 CEO 在周会上发现。
These run after every `dbt run` and in CI before merge — dirty data gets caught before the report, not by the CEO in Monday's meeting.


<a id="8"></a>
## 8. 物化策略 / Materializations

我们的 mini-dbt 支持了 view / table。真 dbt 有四种：
Mini-dbt did view/table; real dbt has four:

| 策略 / Strategy | 是什么 | 用在 / Use for |
|---|---|---|
| **view**（默认）| 只存 SQL，查询时现算 | staging 层（轻逻辑，不值得占存储）|
| **table** | 物化成实体表 | marts 层（被 BI 反复查，算一次用多次）|
| **incremental** ⭐ | 只处理**新增**数据追加进表 | **大事实表**（每天只处理当天分区，不重算 3 年历史）|
| **ephemeral** | 不落库，编译时内联成 CTE | 极轻的中间逻辑 |

### incremental 的样子 / What incremental looks like

```sql
{{ config(materialized='incremental') }}

SELECT ... FROM {{ ref('stg_events') }}
{% if is_incremental() %}
  WHERE event_date > (SELECT MAX(event_date) FROM {{ this }})   -- 只拿新数据
{% endif %}
```

**经验法则**：表小（< 几百万行）→ 无脑 `table` 全量重建，简单可靠；表大 + 追加型（事件日志）→ `incremental`。**过早 incremental 是常见的过度工程**——多了状态就多了 bug 面。
Rule of thumb: small tables → full-rebuild `table` (simple, reliable); big append-only tables → `incremental`. Premature incrementalization is classic over-engineering.


<a id="9"></a>
## 9. 真实 dbt 项目长什么样 / A Real dbt Project

```
my_dbt_project/
├── dbt_project.yml              ← 项目配置（名字、各层默认物化策略）
├── profiles.yml                 ← 连接信息（dev/prod 各一个 target）
├── models/
│   ├── staging/
│   │   ├── _sources.yml         ← 声明 raw 源表 + 源数据新鲜度测试
│   │   ├── stg_invoices.sql
│   │   └── stg_customers.sql
│   ├── intermediate/
│   │   └── int_sales_enriched.sql
│   └── marts/
│       ├── schema.yml           ← 测试 + 文档都写这里
│       ├── fct_monthly_revenue.sql
│       └── dim_customer_summary.sql
├── tests/                       ← 自定义 SQL 测试（singular tests）
├── snapshots/                   ← SCD Type 2 自动化！(1.10 节手写的那套)
└── macros/                      ← 可复用 Jinja 函数
```

### 常用命令 / The commands

```bash
dbt run                          # 物化所有 model（按 DAG 顺序）
dbt test                         # 跑所有测试
dbt build                        # run + test 交错（推荐）
dbt run --select fct_sales+      # 只跑 fct_sales 及其下游（+ 号语法）
dbt docs generate && dbt docs serve    # 生成血缘文档站
```

> 💡 **`snapshots/` 值得一提**：1.10 节我们手写的 SCD Type 2（关旧行 + 插新行）——dbt snapshot **一个配置块全自动**。这是分析工程师爱 dbt 的具体理由之一。
> The SCD Type 2 dance we hand-coded in 1.10? `dbt snapshot` automates it with one config block.


<a id="10"></a>
## 10. 小结 + Part 1 总结 🏁

### dbt 小结 / dbt summary

```
dbt = ELT 的 "T" 的管理工具
  │
  ├── model = 一个 SELECT 文件（无 CREATE，dbt 包）
  ├── ref() ⭐ = 依赖图 + 环境切换 + 血缘，一个机制三件事
  ├── 分层: staging(1:1 清洗) → intermediate(JOIN) → marts(星型成品)
  ├── 测试: unique / not_null / accepted_values / relationships
  ├── 物化: view → table → incremental（别过早）→ ephemeral
  └── snapshot = SCD2 自动化
```

### 💡 面试速查 / Interview must-knows

1. **dbt 解决什么**：依赖排序、测试、版本控制、环境隔离——SQL 的软件工程化
2. **ref() 三合一**：DAG / dev-prod 切换 / lineage
3. **三层职责**：stg 只清洗不 JOIN；int 复用逻辑；marts 出星型
4. **测试四件套** + "0 行 = 通过"的机制
5. **incremental 别过早**：小表全量重建更可靠

---

## 🏁 Part 1 全部完成 / Part 1 Complete!

| # | 课 / Lesson | 核心带走 / Key takeaway |
|---|---|---|
| 1.1 | SQL 基础 | SELECT 管道 + NULL 三件事 |
| 1.2 | JOIN | 六种 JOIN + ON vs WHERE 陷阱 + 行数爆炸 |
| 1.3 | GROUP BY | WHERE vs HAVING + 条件聚合 + ROLLUP/CUBE |
| 1.4 | 子查询 & CTE | correlated 的代价 + WITH RECURSIVE |
| 1.5 | 窗口函数 ⭐ | top-N per group + LAG + 帧子句默认坑 |
| 1.6 | 面试题型 | 10 大模板（sessionization / 留存 / 去重…）|
| 1.7 | 索引 & EXPLAIN | B-tree + 最左前缀 + 慢查询 5 步法 |
| 1.8 | Python ↔ SQL | 参数化防注入 + 连接池 + SQL 干重活 |
| 1.9 | NoSQL | CAP + 四大家族 + "先 Postgres" |
| 1.10 | 数仓 | 星型模型 + 粒度 + SCD2 + ELT |
| 1.11 | dbt | ref() + 分层 + 测试 = SQL 工程化 |

**你现在的 SQL 能力**：覆盖 DS/DA 面试 SQL 轮的全部考点，并懂得它在生产数据栈里的位置。
Your SQL now covers the full DS/DA interview surface — and you know where it sits in the production data stack.

### 下一站 / Next stop

**Part 2 · 统计学与概率** —— 面试第二关。描述统计、假设检验、置信区间、A/B 测试的统计学根基。
**Part 2 · Statistics & Probability** — interview round two: descriptive stats, hypothesis testing, confidence intervals, the statistical foundations of A/B testing.
